<a href="https://colab.research.google.com/github/thddydgnl/NLP_Study/blob/main/NLP_week05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5주차 실습: 텍스트 표현 방법

**주제:** Bag-of-Words(BoW), TF-IDF, n-gram 특징 추출

## 학습 목표
1. 토큰화된 텍스트를 수치 벡터로 표현해야 하는 이유를 설명한다.
2. CountVectorizer로 BoW 행렬을 만들고 희소행렬의 의미를 해석한다.
3. n-gram의 장점과 차원 증가·희소성의 한계를 판단한다.
4. TF-IDF의 TF·DF·IDF를 구분하고 과업에 맞게 설정한다.

> **제출:** 모든 TODO를 작성·실행하고 체크포인트 답변을 Markdown 셀에 남기세요. 외부 AI·자료를 사용했다면 사용 범위와 검증 과정을 기록하세요.

## 0. 준비
교육용 예시 데이터입니다. 실제 데이터는 출처·라이선스·개인정보(PII) 포함 여부를 확인하세요.

In [1]:
# Kiwi가 설치되어 있으면 정식 형태소 분석을 사용합니다.
# Google Colab 등 인터넷 가능한 환경에서 미설치 시 다음 명령을 먼저 실행하세요.
%pip install -q scikit-learn pandas kiwipiepy

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

try:
    from kiwipiepy import Kiwi
    kiwi = Kiwi()
    kiwi_available = True
    print('형태소 분석 환경: Kiwi 사용')
except ImportError:
    kiwi = None
    kiwi_available = False
    print('주의: kiwipiepy 미설치 → 형태소 분석 관련 셀(6·7)은 교육용 대체 토큰화로 실행합니다.')
    print('정확한 Kiwi 형태소 분석 결과는 kiwipiepy 설치 후 노트북 전체를 다시 실행해야 합니다.')

corpus = [
    '배송이 빠르고 제품 품질이 좋아요',
    '배송은 느렸지만 제품 디자인은 예뻐요',
    '품질은 좋지만 가격이 조금 비싸요',
    '가격 대비 품질이 만족스러워요',
    '배송이 너무 느리고 포장도 아쉬워요'
]
df = pd.DataFrame({'doc_id': range(1, len(corpus) + 1), 'text': corpus})
display(df)


Note: you may need to restart the kernel to use updated packages.


형태소 분석 환경: Kiwi 사용


,doc_id,text
0,1,배송이 빠르고 제품 품질이 좋아요
1,2,배송은 느렸지만 제품 디자인은 예뻐요
2,3,품질은 좋지만 가격이 조금 비싸요
3,4,가격 대비 품질이 만족스러워요
4,5,배송이 너무 느리고 포장도 아쉬워요


## 1. 텍스트를 벡터로 표현하는 이유
머신러닝 모델은 문장 자체가 아니라 숫자 입력을 계산합니다. 4주차의 형태소·토큰을 열(feature)로 만들고, 문서마다 각 토큰의 값(빈도 또는 중요도)을 채웁니다.

`원문 → 정제·토큰화 → 문서-토큰 행렬 → 분류/예측`

In [2]:
# TODO 1: 공백 기준 토큰화로 BoW 문서-토큰 행렬 생성
# 기본 CountVectorizer는 정규식 토큰화를 사용하므로, 공백 기준을 명시합니다.
count_vec = CountVectorizer(tokenizer=str.split, token_pattern=None)
X_count = count_vec.fit_transform(corpus)
print('단어 목록:', count_vec.get_feature_names_out())
print('행렬 크기:', X_count.shape, '/ 0이 아닌 원소 수:', X_count.nnz)
display(pd.DataFrame(X_count.toarray(),
                     index=df['doc_id'],
                     columns=count_vec.get_feature_names_out()))


단어 목록: ['가격' '가격이' '너무' '느렸지만' '느리고' '대비' '디자인은' '만족스러워요' '배송은' '배송이' '비싸요' '빠르고'
 '아쉬워요' '예뻐요' '제품' '조금' '좋아요' '좋지만' '포장도' '품질은' '품질이']
행렬 크기: (5, 21) / 0이 아닌 원소 수: 24


,가격,가격이,너무,느렸지만,느리고,대비,디자인은,만족스러워요,배송은,배송이,...,빠르고,아쉬워요,예뻐요,제품,조금,좋아요,좋지만,포장도,품질은,품질이
doc_id,,,,,,,,,,,,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,1,...,1,0,0,1,0,1,0,0,0,1
2,0,0,0,1,0,0,1,0,1,0,...,0,0,1,1,0,0,0,0,0,0
3,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,1,0,1,0
4,1,0,0,0,0,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,1
5,0,0,1,0,1,0,0,0,0,1,...,0,1,0,0,0,0,0,1,0,0


### 체크포인트 1

- **행(row):** 각 행은 입력 리뷰 **한 문서**를 나타내며, 이 실습에서는 1~5번 리뷰에 대응한다.
- **열(column):** 각 열은 전체 문서에서 수집한 하나의 토큰(단어)을 나타낸다.
- **값(value):** 해당 문서에서 그 토큰이 **등장한 횟수**이며, 등장하지 않으면 0이다.
- **0이 많은 이유:** 한 리뷰에는 전체 어휘 중 일부 토큰만 등장하므로 대부분의 문서-토큰 조합은 빈도 0인 희소행렬(sparse matrix)이 된다.

참고로 TODO 1에서는 `str.split`을 지정해 띄어쓰기 단위로 토큰화했기 때문에 `배송이`와 `배송은`은 서로 다른 열로 취급된다.


## 2. Bag-of-Words(BoW)
BoW는 각 단어의 등장 횟수를 사용합니다. 구현은 간단하지만 단어 순서와 긴 문맥 정보를 잃습니다. `배송이 빠르다`와 `빠르지 않은 배송`처럼 순서·부정이 중요한 경우를 생각해 보세요.

In [3]:
# TODO 2: 등장 빈도 기준으로 최대 5개 특징만 남긴 BoW
small_count_vec = CountVectorizer(max_features=5)
X_small = small_count_vec.fit_transform(corpus)
print('선택된 5개 토큰:', small_count_vec.get_feature_names_out())
print('행렬 크기:', X_small.shape)
display(pd.DataFrame(X_small.toarray(),
                     index=df['doc_id'],
                     columns=small_count_vec.get_feature_names_out()))


선택된 5개 토큰: ['배송이' '제품' '좋지만' '포장도' '품질이']
행렬 크기: (5, 5)


,배송이,제품,좋지만,포장도,품질이
doc_id,,,,,
1,1,1,0,0,1
2,0,1,0,0,0
3,0,0,1,0,0
4,0,0,0,0,1
5,1,0,0,1,0


## 3. n-gram: 인접한 토큰 묶기
- unigram: `배송`, `빠르다`
- bigram: `배송 빠르다`
- trigram: `배송 이 빠르다`

n-gram은 일부 어순·표현 정보를 보완하지만, n이 커질수록 특징 수가 늘고 희소성이 커질 수 있습니다.

In [4]:
# TODO 3: unigram과 bigram을 함께 사용
ngram_vec = CountVectorizer(ngram_range=(1, 2))
X_ngram = ngram_vec.fit_transform(corpus)
print('unigram + bigram 특징:', ngram_vec.get_feature_names_out())
print('특징 수:', X_ngram.shape[1])
print('unigram만의 특징 수:', CountVectorizer().fit_transform(corpus).shape[1])


unigram + bigram 특징: ['가격' '가격 대비' '가격이' '가격이 조금' '너무' '너무 느리고' '느렸지만' '느렸지만 제품' '느리고'
 '느리고 포장도' '대비' '대비 품질이' '디자인은' '디자인은 예뻐요' '만족스러워요' '배송은' '배송은 느렸지만' '배송이'
 '배송이 너무' '배송이 빠르고' '비싸요' '빠르고' '빠르고 제품' '아쉬워요' '예뻐요' '제품' '제품 디자인은'
 '제품 품질이' '조금' '조금 비싸요' '좋아요' '좋지만' '좋지만 가격이' '포장도' '포장도 아쉬워요' '품질은'
 '품질은 좋지만' '품질이' '품질이 만족스러워요' '품질이 좋아요']
특징 수: 40
unigram만의 특징 수: 21


### 체크포인트 2

감성 분석에서는 `너무 느리고`와 같은 bigram이 unigram `너무`, `느리고` 각각보다 유용할 수 있다. 인접한 두 토큰을 묶으면 **부정적인 평가에 대한 강도**를 함께 표현할 수 있기 때문이다. 예를 들어 형태소 기반으로 `안 좋`을 보존하면 `좋`만 봤을 때 발생할 수 있는 부정 표현 오해를 줄일 수 있다.

다만 bigram을 추가하면 특징 차원이 커지고 각 표현의 관측 횟수가 줄어 **행렬이 더 희소해진다**. 데이터가 적을 때는 우연히 등장한 표현까지 학습하여 과적합하고, 메모리·계산 비용이 늘 수 있으므로 특징 수 증가가 항상 성능 향상을 뜻하지는 않는다.


## 4. TF-IDF: 많이 나오지만 덜 구별되는 단어의 가중치 낮추기
- **TF (Term Frequency):** 한 문서 안에서의 단어 빈도
- **DF (Document Frequency):** 해당 단어를 포함한 문서 수
- **IDF (Inverse Document Frequency):** 여러 문서에 흔한 단어의 가중치를 낮추는 값

TF-IDF는 단어의 중요도를 표현하는 한 방법이며, 과업에 따라 BoW가 더 적절할 수도 있습니다.

In [5]:
# TODO 4: TF-IDF 행렬 (기본 설정: 평활 IDF, L2 정규화)
tfidf_vec = TfidfVectorizer()
X_tfidf = tfidf_vec.fit_transform(corpus)
tfidf_df = pd.DataFrame(X_tfidf.toarray(),
                        index=df['doc_id'],
                        columns=tfidf_vec.get_feature_names_out())
print('TF-IDF 행렬 크기:', X_tfidf.shape)
display(tfidf_df.round(2))


TF-IDF 행렬 크기: (5, 21)


,가격,가격이,너무,느렸지만,느리고,대비,디자인은,만족스러워요,배송은,배송이,...,빠르고,아쉬워요,예뻐요,제품,조금,좋아요,좋지만,포장도,품질은,품질이
doc_id,,,,,,,,,,,,,,,,,,,,,
1,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.41,...,0.5,0.00,0.00,0.41,0.00,0.5,0.00,0.00,0.00,0.41
2,0.00,0.00,0.00,0.46,0.00,0.00,0.46,0.00,0.46,0.00,...,0.0,0.00,0.46,0.37,0.00,0.0,0.00,0.00,0.00,0.00
3,0.00,0.45,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.0,0.00,0.00,0.00,0.45,0.0,0.45,0.00,0.45,0.00
4,0.52,0.00,0.00,0.00,0.00,0.52,0.00,0.52,0.00,0.00,...,0.0,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.42
5,0.00,0.00,0.46,0.00,0.46,0.00,0.00,0.00,0.00,0.37,...,0.0,0.46,0.00,0.00,0.00,0.0,0.00,0.46,0.00,0.00


In [6]:
# TODO 5: 2번 문서에서 BoW와 TF-IDF의 상위 3개 토큰 비교
target_idx = 1  # 0-based 인덱스이므로 두 번째 리뷰
print('대상 문서:', corpus[target_idx])
count_scores = pd.Series(X_count.toarray()[target_idx],
                         index=count_vec.get_feature_names_out())
tfidf_scores = pd.Series(X_tfidf.toarray()[target_idx],
                         index=tfidf_vec.get_feature_names_out())
print('BoW 상위 3개 (빈도 동률이면 순위에 큰 의미가 없음):')
print(count_scores.sort_values(ascending=False).head(3))
print('TF-IDF 상위 3개:')
print(tfidf_scores.sort_values(ascending=False).head(3).round(3))
print('공통 토큰의 가중치 비교 (등장한 토큰만):')
comparison = pd.DataFrame({'BoW': count_scores,
                           'TF-IDF': tfidf_scores}).fillna(0)
display(comparison[comparison['BoW'] > 0].sort_values('TF-IDF', ascending=False).round(3))


대상 문서: 배송은 느렸지만 제품 디자인은 예뻐요
BoW 상위 3개 (빈도 동률이면 순위에 큰 의미가 없음):
느렸지만    1
디자인은    1
배송은     1
dtype: int64
TF-IDF 상위 3개:
느렸지만    0.464
디자인은    0.464
배송은     0.464
dtype: float64
공통 토큰의 가중치 비교 (등장한 토큰만):


,BoW,TF-IDF
느렸지만,1,0.464
디자인은,1,0.464
배송은,1,0.464
예뻐요,1,0.464
제품,1,0.374


### 체크포인트 3

**여러 문서에 공통으로 등장하는 단어는 DF가 높으므로 IDF가 상대적으로 낮아져 TF-IDF 가중치가 작아진다.** 반대로 특정 문서에만 나타나는 토큰은 더 높은 IDF를 받을 수 있다. 단, 실제 TF-IDF 값은 한 문서의 빈도(TF)와 L2 정규화에도 영향을 받는다. scikit-learn의 기본 평활 IDF는 `log((1 + 전체 문서 수) / (1 + DF)) + 1`이다.

**중요할 수 있는 사례:** 상품 리뷰 **감성 분석**에서는 여러 리뷰에 반복 등장하는 `좋다`나 `나쁘다`가 실제로는 감성을 구분하는 핵심 단서일 수 있다. 따라서 흔한 단어라는 이유만으로 항상 제거해서는 안 된다.

참고: TODO 1의 띄어쓰기 토큰화에서는 조사 차이 때문에 `품질이`와 `품질은`이 다른 열이지만, 형태소 분석을 하면 둘을 공통 명사 `품질`로 묶을 수 있다.


## 5. 한국어 형태소 토큰을 벡터화하기
공백 토큰화 대신 4주차의 형태소 분석 결과를 벡터화할 수 있습니다. 여기서는 키워드 추출 예시로 명사 계열을 사용합니다. 감성 분석이라면 용언·부사·부정 표현 보존을 추가로 검토해야 합니다.

In [7]:
# Kiwi가 설치되면 명사 품사(N*)를 추출합니다.
# 미설치 환경에서는 예시 말뭉치에 한정한 사전 기반 대체 함수를 사용합니다.
# 아래 대체 함수는 **형태소 분석이 아니며**, Kiwi 분석 성능을 검증하지 않습니다.
NOUN_FALLBACK_VOCAB = ('배송', '제품', '품질', '디자인', '가격', '포장')

def kiwi_noun_tokens(text):
    if kiwi is not None:
        return [token.form for token in kiwi.tokenize(text)
                if token.tag.startswith('N') and len(token.form) > 1]
    # 오프라인 환경의 교육용 간이 대체: 알려진 명사가 앞에 오는 공백 토큰만 탐지
    return [noun for part in text.split()
            for noun in NOUN_FALLBACK_VOCAB if part.startswith(noun)]

print('토큰화 모드:', 'Kiwi 명사 형태소 분석' if kiwi_available else '예시 어휘 사전 기반 대체')
print('예시 토큰:', kiwi_noun_tokens(corpus[0]))

# TODO 6: 명사 토크나이저를 사용한 TF-IDF
korean_tfidf = TfidfVectorizer(tokenizer=kiwi_noun_tokens, token_pattern=None)
X_korean = korean_tfidf.fit_transform(corpus)
print('특징:', korean_tfidf.get_feature_names_out())
display(pd.DataFrame(X_korean.toarray(),
                     index=df['doc_id'],
                     columns=korean_tfidf.get_feature_names_out()).round(2))


토큰화 모드: Kiwi 명사 형태소 분석
예시 토큰: ['배송', '제품', '품질']
특징: ['가격' '대비' '디자인' '만족' '배송' '제품' '포장' '품질']


,가격,대비,디자인,만족,배송,제품,포장,품질
doc_id,,,,,,,,
1,0.00,0.00,0.00,0.00,0.54,0.65,0.00,0.54
2,0.00,0.00,0.69,0.00,0.46,0.56,0.00,0.00
3,0.77,0.00,0.00,0.00,0.00,0.00,0.00,0.64
4,0.46,0.57,0.00,0.57,0.00,0.00,0.00,0.38
5,0.00,0.00,0.00,0.00,0.56,0.00,0.83,0.00


## 6. 미니 과제: 특징 추출 정책 설계
다음 중 하나를 선택하세요. (1) 강의평 키워드 추출 (2) 상품 리뷰 감성 분석 (3) 학교 민원 자동 분류

분석기/토큰 단위, 품사 선택, n-gram 범위, 최소 문서 빈도(`min_df`), 사용할 표현(BoW 또는 TF-IDF)을 근거와 함께 결정하세요.

In [8]:
# TODO 7: 과업 (2) 상품 리뷰 감성 분석
# 명사(N*), 동사·형용사·보조용언(V*), 일반·접속부사(MAG/MAJ)를 보존하여
# 품질/가격 등의 대상어와 긍정·부정 단서('좋', '느리', '안' 등)를 함께 담습니다.
def review_tokens(text):
    if kiwi is not None:
        return [token.form for token in kiwi.tokenize(text)
                if token.tag.startswith(('N', 'V')) or token.tag in {'MAG', 'MAJ'}]
    # 대체 모드: 형태소·품사를 구별하지 못하므로 공백 토큰을 그대로 사용
    return text.split()

my_vectorizer = TfidfVectorizer(
    tokenizer=review_tokens,
    token_pattern=None,
    ngram_range=(1, 2),  # 단일 토큰과 인접 두 토큰 표현 모두 유지
    min_df=1             # 예시 5문서에서 드문 감성 표현을 누락하지 않도록 설정
)
X_my = my_vectorizer.fit_transform(corpus)
print('과업: 상품 리뷰 감성 분석')
print('토큰화 모드:', 'Kiwi (N/V/MAG/MAJ)' if kiwi_available else '공백 토큰 대체 - Kiwi 재실행 필요')
print('특징 수:', X_my.shape[1])
print('특징:', my_vectorizer.get_feature_names_out())
display(pd.DataFrame(X_my.toarray(),
                     index=df['doc_id'],
                     columns=my_vectorizer.get_feature_names_out()).round(2))


과업: 상품 리뷰 감성 분석
토큰화 모드: Kiwi (N/V/MAG/MAJ)
특징 수: 34
특징: ['가격' '가격 대비' '가격 조금' '너무' '너무 느리' '느리' '느리 제품' '느리 포장' '대비' '대비 품질' '디자인'
 '디자인 예쁘' '만족' '배송' '배송 너무' '배송 느리' '배송 빠르' '비싸' '빠르' '빠르 제품' '아쉽' '예쁘'
 '제품' '제품 디자인' '제품 품질' '조금' '조금 비싸' '좋' '좋 가격' '포장' '포장 아쉽' '품질' '품질 만족'
 '품질 좋']


,가격,가격 대비,가격 조금,너무,너무 느리,느리,느리 제품,느리 포장,대비,대비 품질,...,제품 품질,조금,조금 비싸,좋,좋 가격,포장,포장 아쉽,품질,품질 만족,품질 좋
doc_id,,,,,,,,,,,,,,,,,,,,,
1,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0,...,0.38,0.00,0.00,0.31,0.00,0.00,0.00,0.26,0.0,0.31
2,0.00,0.0,0.00,0.00,0.00,0.29,0.36,0.00,0.0,0.0,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.00
3,0.30,0.0,0.37,0.00,0.00,0.00,0.00,0.00,0.0,0.0,...,0.00,0.37,0.37,0.30,0.37,0.00,0.00,0.25,0.0,0.30
4,0.33,0.4,0.00,0.00,0.00,0.00,0.00,0.00,0.4,0.4,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.27,0.4,0.00
5,0.00,0.0,0.00,0.35,0.35,0.28,0.00,0.35,0.0,0.0,...,0.00,0.00,0.00,0.00,0.00,0.35,0.35,0.00,0.0,0.00


### 체크포인트 4 — 특징 추출 정책

**선택한 과업: (2) 상품 리뷰 감성 분석**

1. **BoW/TF-IDF 선택:** `TfidfVectorizer`를 사용한다. 단순 빈도(BoW)뿐 아니라 문서 간 구별력(IDF)을 반영할 수 있기 때문이다. 다만 감성에서 자주 등장하는 중요한 단어가 낮은 IDF를 가질 수 있으므로, 실제 분류 성능은 BoW와 비교 검증해야 한다.
2. **토큰·품사 선택:** Kiwi 분석의 명사(N*)로 평가 대상(배송·제품·품질·가격)을, 동사/형용사/보조용언(V*)으로 감성 단서(좋·느리·비싸·아쉽)를, 부사(MAG/MAJ)로 정도·부정 표현(너무·안·못)을 보존한다. 조사·어미는 기본적으로 제외한다. `kiwi_noun_tokens`와 달리 감성 분석용 `review_tokens`는 한 글자 단서도 제거하지 않는다.
3. **n-gram 및 `min_df`:** `ngram_range=(1, 2)`로 unigram과 bigram을 함께 사용해 `너무 느리`, `안 좋` 같은 결합 표현을 표현하도록 한다. `min_df=1`은 리뷰가 5개뿐이므로 희귀 감성 표현을 버리지 않기 위한 설정이다. 자료가 충분해지면 `min_df`와 n-gram 범위를 검증 데이터 성능으로 조정한다.
4. **예상 오류·정보 손실:** `안 좋다`와 `좋다`를 unigram만으로 구별하기 어렵고, 조사를 제거하면서 의미 관계가 유실될 수 있다. 또한 `안 정말 좋다`처럼 거리가 있는 부정은 bigram으로 포착하지 못할 수 있다. 부정어가 제거되거나 문장 순서가 바뀌어도 비슷한 특징이 생성되는 오류, 신조어·맞춤법 변형으로 인한 형태소 분석 오류도 가능하다.

## 제출 체크리스트
- [x] TODO 1~7의 코드를 모두 작성하고 모든 코드 셀을 실행했다.
- [x] BoW 행렬의 행·열·값·희소성을 설명했다.
- [x] unigram과 bigram의 차이·한계를 설명했다.
- [x] TF·DF·IDF의 역할을 설명했다.
- [x] 공백 기반과 형태소 기반 벡터화의 차이를 설명했다.
- [x] 선택 과업에 맞는 특징 추출 정책과 오류 사례를 기록했다.
- [x] 외부 AI/자료 사용 범위와 검증 과정을 기록했다.

### 외부 AI 사용 및 검증 기록

> 추가 안내 (2026-11-09): 기존 과제 기간 내에 전부 제출을 완료했으나, 공부를 하던 중 AI 사용 기록이 충분하지 않은 것을 알게 되어 AI 사용 기록 부분을 보완했습니다. 제출 당시에는 kiwipiepy가 설치되지 않아 TODO 6·7이 대체 토큰화로 실행되었기 때문에, 이번에 kiwipiepy를 설치해 노트북을 다시 실행하고 이를 알리던 '실행 환경 유의' 문단을 삭제했습니다.

- 사용 도구: ChatGPT, Claude
- 사용 범위:
  - TODO 1~7 코드: 먼저 직접 작성한 뒤 AI에게 검토를 받았고, 오류가 있을 때 해결 방법을 물어 고쳤습니다. TODO 6에서 kiwipiepy가 없어 오류가 났을 때는 ChatGPT가 알려 준 대로 kiwipiepy가 없으면 대체 토큰화로 실행되도록 고쳤습니다.
  - 체크포인트 1~4 답변: 먼저 스스로 생각한 뒤 AI에게 제 생각과 다른 관점을 물어 공부했고, 답변은 직접 쓴 다음 AI로 문장을 다듬었습니다.
- 검증 과정:
  - AI가 고쳐 준 코드는 다시 실행해 오류가 사라지고 출력이 나오는지 확인했습니다. 다만 TODO 6의 대체 토큰화는 Kiwi 결과가 아니어서, 기록을 보완하면서 kiwipiepy를 설치해 다시 실행하고 TODO 6·7이 Kiwi 토큰화 모드로 실행되어 TODO 7 특징이 형태소 단위로 바뀐 것을 확인했습니다.
  - AI로 다듬은 답변은 제가 쓴 뜻과 실행 출력에 맞는지 대조했습니다. 예: 체크포인트 3의 IDF 설명을 TODO 5 출력과 대조(두 문서에 나오는 `제품`의 TF-IDF 0.374가 한 문서에만 나오는 토큰의 0.464보다 낮음)
  - AI가 알려 준 다른 관점과 설명은 강의자료나 검색 결과와 비교했습니다.

## 다음 주차 예고
다음 시간에는 만들어진 문서-토큰 행렬을 입력으로 사용하여 나이브 베이즈와 로지스틱 회귀 기반 텍스트 분류 모델을 학습하고 평가합니다.
